<a href="https://colab.research.google.com/github/sitthinon-stat/Sports-court-rental/blob/rental-1/Sports_court_rental.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sports-court-rental
ไว้สำหรับการจองสนามกีฬา จาก Project อาจารย์พิชญา


## ส่วนที่ 1 — เตรียม class และฟังก์ชัน

In [8]:
import random
import time
random.seed(7)
import datetime
import pandas as pd

In [9]:
class Customer:
#class 1 ลูกค้าจองสนามกีฬา
  def __init__(self, customer_id, name, phone_number, is_member=False):
    self.customer_id = customer_id
    self.name = name
    self.phone_number = phone_number
    self.is_member = is_member

  def register_member(self):
    self.is_member = True
    print(f'{self.name} สมัครสมาชิกสำเร็จ><')

class Court:
#class 2 สนามกีฬา
  def __init__(self, court_id, sport_type, hourly_rate, court_status=True):
    self.court_id = court_id
    self.sport_type = sport_type
    self.hourly_rate = hourly_rate
    self.court_status = court_status

  def set_court_unavailable(self):
    self.court_status = False
    print(f'สนาม {self.court_id} ถูกจองหรือปิดปรับปรุง')

class Equipment:
#class 3 อุปกรณ์เสริมให้เช่า
  def __init__(self, equipment_id, equipment_name, equipment_price, stock_quantity):
    self.equipment_id = equipment_id
    self.equipment_name = equipment_name
    self.equipment_price = equipment_price  # Corrected typo: prive -> price
    self.stock_quantity = stock_quantity

  def reduce_stock(self, amount):
    if self.stock_quantity >= amount:
      self.stock_quantity-= amount
      return True
    print(f"อุปกรณ์ {self.equipment_name} หมด")
    return False

  def return_stock(self, amount):
    self.stock_quantity += amount

class Booking:
#class 4 การจองสนาม
  def __init__(self, booking_id, customer, court, start_time, hours):
    self.booking_id = booking_id
    self.customer = customer
    self.court = court
    self.start_time = start_time
    self.hours = hours
    self.equipment_list = []
    self.status = "Confirm"

  def add_equipment(self, equipment, quantity):
    if equipment.reduce_stock(quantity):
        self.equipment_list.append((equipment, quantity))
        print(f"เพิ่ม {equipment.equipment_name} จำนวน {quantity} ชิ้น ในการจองแล้ว")

  def calculate_total_price(self):
        court_price = self.court.hourly_rate * self.hours
        equipment_price = sum(
            item.equipment_price * qty for item, qty in self.equipment_list # Corrected: item.rental_price -> item.equipment_price
        )

        total = court_price + equipment_price
        return total

class Payment:
#class 5 การชำระเงิน
  def __init__(self, payment_id, booking,  payment_method):
    self.payment_id = payment_id
    self.booking = booking
    self.payment_method = payment_method
    # The amount is calculated here, so it relies on Booking.calculate_total_price
    self.amount = self.booking.calculate_total_price()
    self.payment_status = "Pending"

  def process_payment(self):
        self.payment_status = "Paid"
        print(
            f"ชำระเงินสำเร็จ: รหัส {self.payment_id} ยอดเงิน {self.amount} บาท ผ่าน {self.payment_method}"
        )

In [12]:
def generate_thai_name():
    # ฟังก์ชัน: สุ่มชื่อ-นามสกุลลูกค้า -> คืนค่าเป็น string
    first_names = [
        "สมชาย", "สมหญิง", "วิชัย", "อรุณี", "ปรีชา",
        "มานี", "กิตติ", "ศิริพร", "ธนากร", "นภัสสร",
        "ธีรภัทร", "ณัฐวุฒิ", "สุพรรษา", "วรวิทย์", "ชลธิชา","ชินานาง","สิทธินนท์"
    ]

    last_names = [
        "ใจดี", "รักเรียน", "สายทอง", "ศรีสุข", "มั่นคง",
        "เจริญสุข", "วงศ์สว่าง", "รัตนไพศาล", "สุขเกษม", "พัฒนากุล",
        "แสงสุริยา", "ทองประเสริฐ", "เกียรติขจร", "ประสิทธิ์โชค", "บุญรักษา","เชื้อกุณะ","โยธาธรณ์"
    ]

    return f"{random.choice(first_names)} {random.choice(last_names)}"


def generate_phone_number():
    # ฟังก์ชันเสริม: สุ่มเบอร์โทรศัพท์มือถือไทย 10 หลัก เพื่อใช้สร้าง Customer
    prefixes = ["081", "086", "089", "092", "095", "061"]
    suffix = "".join([str(random.randint(0, 9)) for _ in range(7)])
    prefix = random.choice(prefixes)
    return f"{prefix}-{suffix[:3]}-{suffix[3:]}"


def random_booking_hours(min_hours=1, max_hours=4):
    # ฟังก์ชัน: สุ่มจำนวนชั่วโมงที่ต้องการจองสนาม (1 - 4 ชั่วโมง) -> คืนค่าเป็น int
    return random.randint(min_hours, max_hours)


def random_start_time():
   # ฟังก์ชันเสริม: สุ่มเวลาเริ่มเล่น (ช่วง 08:00 - 21:00 น.)
    hour = random.randint(8, 21)
    minute = random.choice(["00", "30"])
    return f"{hour:02d}:{minute}"


def format_currency(amount, symbol="บาท"):
    # ฟังก์ชัน: จัดรูปแบบตัวเลขเป็นสตริงราคาพร้อมเครื่องหมายจุลภาค -> คืนค่าเป็น string
    return f"{amount:,.2f} {symbol}"

## ส่วนที่ 2 — ทดสอบฟังก์ชันทีละตัว ก่อนเอาไปประกอบเป็นกระบวนการ

ก่อนจะเอาฟังก์ชันไปใช้จริง ควรลองเรียกดูเฉย ๆ ทีละตัวก่อน เพื่อดูว่า **input ที่ใส่เข้าไป**
กับ **output ที่ได้กลับมา** ตรงกับที่ออกแบบไว้หรือไม่


In [24]:
# เรียก generate_thai_name() 8 ครั้ง -> ทุกครั้งได้ชื่อสุ่มไม่ซ้ำแบบ (แสดงว่าฟังก์ชันทำงานทุกครั้งที่เรียก)
for _ in range(10):
    print("ชื่อที่สุ่มได้:", generate_thai_name())

ชื่อที่สุ่มได้: สมชาย สายทอง
ชื่อที่สุ่มได้: ชลธิชา แสงสุริยา
ชื่อที่สุ่มได้: สิทธินนท์ โยธาธรณ์
ชื่อที่สุ่มได้: กิตติ สุขเกษม
ชื่อที่สุ่มได้: ชลธิชา โยธาธรณ์
ชื่อที่สุ่มได้: ชินานาง โยธาธรณ์
ชื่อที่สุ่มได้: ศิริพร โยธาธรณ์
ชื่อที่สุ่มได้: ธนากร วงศ์สว่าง
ชื่อที่สุ่มได้: ชลธิชา มั่นคง
ชื่อที่สุ่มได้: วรวิทย์ ศรีสุข
